# Pipeline E — Evaluation Notebook

Runs `evaluate_all.py` on the held-out 15% test split using the LoRA checkpoint trained in the previous Colab session.

**Prereqs:**
- Google Colab runtime with **GPU** (A100 strongly recommended, ~2.5h runtime)
- LoRA checkpoint present at `/content/drive/MyDrive/pipeline_e_checkpoints/seed_0/lora/`
- ~100 compute units available on Colab Pro

**Outputs (saved to Drive):**
- `benchmark_table.csv` — test metrics for all 5 pipelines
- `confusion_matrices.png`
- `calibration_curves.png`
- `umap_embeddings.png`
- Cross-domain SPS evaluation (printed)

## § 1 — Setup

In [ ]:
# Mount Drive so checkpoints + eval results persist across runtime restarts.
# /content/ is wiped on disconnect; /content/drive/MyDrive/ is permanent.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Pull the latest project code. rm -rf ensures a clean clone if the folder lingers from a previous session.
%cd /content
!rm -rf EE-519-Project
!git clone https://github.com/Ncostanz05/EE-519-Project.git
%cd EE-519-Project
!ls

In [ ]:
# Exact versions the project was trained against.
# numpy<2 + pandas==2.2.2 avoid the ABI 'numpy.dtype size changed 96 vs 88' error.
# peft = LoRA; umap/scipy = eval plots + metrics; transformers = WavLM backbone.
# parselmouth + joblib needed for Pipeline A (features.py imports them at module level).
!pip install -q "numpy<2" "pandas==2.2.2"
!pip install -q peft==0.10.0 umap-learn==0.5.6 scipy==1.13.0 accelerate==0.29.3
!pip install -q transformers==4.40.2 torch==2.2.2 torchaudio==2.2.2 \
    librosa==0.10.1 scikit-learn==1.4.2 pandas==2.2.2
!pip install -q praat-parselmouth==0.4.3 joblib==1.4.2

In [ ]:
# Quick sanity: Colab sometimes silently assigns CPU. Confirm GPU is live AND able to run a tensor op.
# If either fails -> Runtime -> Change runtime type -> GPU -> Reconnect.
import torch
print(f"torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")

try:
    x = torch.randn(100, 100).cuda()
    y = x @ x
    print(f"GPU compute works: {y.sum().item():.2f}")
except Exception as e:
    print(f"GPU broken: {e}")

## § 2 — Data setup

Skip this section if a previous session on the same runtime already extracted the dataset to `/content/cv-data/`.
After a runtime restart, `/content/` is wiped and these cells must run again.

In [ ]:
# Cheap check - skip the 88GB download if a previous session already extracted everything.
import os
!ls /content/cv-data/ 2>/dev/null || echo "cv-data not found"
!du -sh /content/cv-data 2>/dev/null

In [ ]:
# Pull the 88GB CV-en-v25 tarball from the Mozilla CDN (Cloudflare R2).
# !! The signed URL expires after 12h. Regenerate a fresh URL from commonvoice.mozilla.org before running.
# Download takes ~30-40 min depending on bandwidth.
CV_URL = "PASTE_FRESH_SIGNED_URL_HERE"
!wget -O /content/cv-en.tar.gz "$CV_URL"
!ls -lh /content/cv-en.tar.gz

In [ ]:
# Unpack ~88GB of MP3s + metadata into /content/cv-data/. Takes ~20-30 min.
# Immediately rm the tarball to free ~90GB of Colab disk.
!mkdir -p /content/cv-data
!tar -xzf /content/cv-en.tar.gz -C /content/cv-data/
!rm /content/cv-en.tar.gz
!ls /content/cv-data/

In [ ]:
# Confirm the expected paths exist. en/clips holds the MP3 files; validated.tsv is the label manifest.
!ls /content/cv-data/cv-corpus-25.0-2026-03-09/en/
!ls /content/cv-data/cv-corpus-25.0-2026-03-09/en/clips | head -5
!du -sh /content/cv-data/

In [ ]:
# Recreate the EXACT 100K-sample subset used for training (random_state=42).
# The test split is deterministic on (DataFrame, seed=42) so the subset must match
# what training saw, or the 'test set' will include samples the model was trained on.
import os
import pandas as pd

CV_ROOT = "/content/cv-data/cv-corpus-25.0-2026-03-09/en"
n_clips = len(os.listdir(f"{CV_ROOT}/clips"))
print(f"Extracted clips: {n_clips}")

df = pd.read_csv(f"{CV_ROOT}/validated.tsv", sep="\t", low_memory=False)
df_labeled = df[df['age'].notna() & df['gender'].notna()]
print(f"Labeled in TSV: {len(df_labeled)}")

existing = set(os.listdir(f"{CV_ROOT}/clips"))
df_labeled_on_disk = df_labeled[df_labeled['path'].isin(existing)]
print(f"Labeled AND extracted: {len(df_labeled_on_disk)}")

df_sample = df_labeled_on_disk.sample(n=min(100_000, len(df_labeled_on_disk)), random_state=42)
df_sample.to_csv(f"{CV_ROOT}/commonvoice_subset.csv", index=False)
print(f"\nSubset saved: {len(df_sample)} rows")

## § 3 — Checkpoint verification

In [ ]:
# The LoRA checkpoint from the previous training session should be on Drive.
# best_model.pt = WavLM-large + LoRA adapters + heads (~1.27GB)
# calibrator.pt = temperature scalar learned after training (~1KB)
import os
ckpt_dir = "/content/drive/MyDrive/pipeline_e_checkpoints/seed_0/lora"
for f in ["best_model.pt", "calibrator.pt"]:
    p = f"{ckpt_dir}/{f}"
    size = os.path.getsize(p) / 1e6 if os.path.exists(p) else 0
    print(f"  {f}: {'[OK]' if os.path.exists(p) else '[MISSING]'}  {size:.1f} MB")

In [ ]:
# Show the best epoch + val macro-F1 the checkpoint was saved at.
# Lightweight: doesn't instantiate the model, just reads the metadata dict.
# On torch 2.2.2 weights_only defaults to False and handles the PipelineEConfig dataclass natively.
%cd /content/EE-519-Project
import torch, os

ckpt = torch.load(os.path.join(ckpt_dir, "best_model.pt"), map_location="cpu")
print(f"Best epoch:   {ckpt.get('epoch')}")
print(f"Val macro-F1: {ckpt.get('val_macro_f1'):.4f}")

In [ ]:
# Full training summary (both frozen and LoRA stages across all seeds).
# Pure stdlib -- works even if torch/numpy are misbehaving.
import json
with open("/content/drive/MyDrive/pipeline_e_checkpoints/training_summary.json") as f:
    print(json.dumps(json.load(f), indent=2))

## § 4 — Evaluation

In [ ]:
# Smoke test on 100 test samples (~5 min, ~1 compute unit).
# Verifies the script loads all pipelines and runs end-to-end before committing to the full 2.5h eval.
# If this fails, fix the issue before Cell below.
!python evaluate_all.py \
    --pe-checkpoint /content/drive/MyDrive/pipeline_e_checkpoints/seed_0/lora \
    --cv-csv /content/cv-data/cv-corpus-25.0-2026-03-09/en/commonvoice_subset.csv \
    --cv-audio-dir /content/cv-data/cv-corpus-25.0-2026-03-09/en/clips \
    --limit 100

In [ ]:
# MAIN DELIVERABLE: full test-set evaluation of all 5 pipelines.
# Pipelines A-D auto-skip if their .pkl/.pth files aren't in the repo root.
# Writes to eval_results/: benchmark_table.csv, confusion_matrices.png, calibration_curves.png, umap_embeddings.png
# Also runs SPS cross-domain eval if sps-corpus-1.0-2025-11-25-en/ is present.
# Runtime: ~2-2.5h on A100 (~32 compute units). These numbers go in your writeup.
!python evaluate_all.py \
    --pe-checkpoint /content/drive/MyDrive/pipeline_e_checkpoints/seed_0/lora \
    --cv-csv /content/cv-data/cv-corpus-25.0-2026-03-09/en/commonvoice_subset.csv \
    --cv-audio-dir /content/cv-data/cv-corpus-25.0-2026-03-09/en/clips

## § 5 — Save results to Drive

In [ ]:
# eval_results/ lives under /content/ and gets wiped with the runtime.
# Mirror it to Drive so tables/plots survive and you can download them from any machine.
!cp -r eval_results /content/drive/MyDrive/
!ls -la /content/drive/MyDrive/eval_results/

In [ ]:
# Preview the headline benchmark table inline.
import pandas as pd
df_bench = pd.read_csv("/content/drive/MyDrive/eval_results/benchmark_table.csv")
df_bench

In [ ]:
# Display the confusion matrices + calibration curves + UMAP inline for quick review.
from IPython.display import Image, display
for name in ["confusion_matrices.png", "calibration_curves.png", "umap_embeddings.png"]:
    path = f"/content/drive/MyDrive/eval_results/{name}"
    import os
    if os.path.exists(path):
        print(f"\n=== {name} ===")
        display(Image(path))